# 06 · Illustrative EL decomposition and PSI
## Part A - EL = PD x LGD x EAD (**proxies, not production models**)
- **EAD proxy:** balance at default `= funded_amnt - total_rec_prncp` on charged-off loans; CCF_proxy = mean(EAD / funded_amnt). Term loans are fully drawn at origination, so this "CCF" is really amortisation.
- **LGD proxy:** exposure-weighted `1 - recoveries / EAD`, clipped to [0,1], on **train** defaults.
- These use `funded_amnt`, `recoveries`, `total_rec_prncp`, which are **banned from the PD feature set** (post-origination / outcome-linked). They are used only here.
- Limits: recoveries are undiscounted, no downturn LGD, no workout-cost data, no segmentation. A bank would model LGD with beta/fractional regression or a two-stage model and discount recoveries at the contract rate.

In [ ]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from src.utils.config import load_config, set_seeds
cfg = load_config(); set_seeds(cfg["seed"])
pd.set_option("display.float_format", "{:.4f}".format); pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40)
print("config seed:", cfg["seed"], "| raw file:", cfg["paths"]["raw_csv"])

In [ ]:
from src.pipeline import prepare, get_models
from src.models.el_proxy import fit_lgd_ead_proxy
data = prepare(cfg); models = get_models(data, cfg); champ = models["champion_logit"]
td = data.train[data.train.target==1]
el = fit_lgd_ead_proxy(td); print(el)
d = td.assign(ead=(td.funded_amnt-td.total_rec_prncp).clip(lower=0))
d["lgd_i"] = (1 - d.recoveries/d.ead.where(d.ead>0)).clip(0,1)
print("LGD distribution on train defaults:"); print(d.lgd_i.describe().round(3).to_string())

In [ ]:
v = data.valid.assign(pd_hat=champ.predict(data.valid))
v["EL"] = v.pd_hat * el["lgd"] * el["ccf"] * v.loan_amnt
bands = pd.qcut(v.pd_hat, 5, labels=[f"Q{i}" for i in range(1,6)])
g = v.groupby(bands, observed=True).agg(loans=("pd_hat","size"), mean_pd=("pd_hat","mean"), observed_dr=("target","mean"), exposure=("loan_amnt","sum"), EL=("EL","sum"))
g["EL_rate"] = g.EL/g.exposure; g

## Part B - PSI (distribution shift, **not** performance)
PSI compares the distribution of a score or feature in a *reference* sample (train) to another sample. It uses **no outcomes**, so:
- low PSI does **not** mean the model performs well; high PSI does **not** mean it performs badly;
- it is a *monitoring trigger* ("has the population moved?"), to be read next to the performance/calibration results, never instead of them.

Convention (not a theorem): <0.10 stable, 0.10-0.25 moderate shift, >0.25 major shift. 2017-2018 loans are included here (any status) because PSI needs features, not outcomes.

In [ ]:
from src.pipeline import psi_tables
pt = psi_tables(models, data)
pt["score"]

In [ ]:
pt["features"].style.format("{:.3f}").background_gradient(cmap="Reds", vmin=0, vmax=0.25)

In [ ]:
s = pt["score"].set_index("sample").score_psi
fig, ax = plt.subplots(figsize=(6,3.6)); ax.bar(s.index, s.values, color="#72b7b2"); ax.axhline(.1,color="orange",ls=":"); ax.axhline(.25,color="r",ls=":")
ax.set_title("Score PSI vs train (distribution shift only)"); plt.xticks(rotation=30); plt.show()